# Actividad grupal clase 3 — Cadenas y procesos de Markov

**BCD5105 · Modelado matemático · III Cuatrimestre 2026 · Lead University**

**Integrantes del grupo:** _(listar aquí a los integrantes)_

> Este notebook usa el **entorno de R** (Colab: *Entorno de ejecución → Cambiar tipo de entorno → R*, o `colab.to/r`). Las celdas de código llevan como comentario la salida que debe aparecer.

Contenido:
1. Miniexploración a mano: el cliente de un banco digital
2. La cadena en R (`markovchain`)
3. Una campaña de retención
4. PageRank de un sitio de trámites (`igraph`)
5. Reflexión y tema del proyecto

In [ ]:
# La instalación tarda uno o dos minutos: correr esta celda apenas se abra el notebook
install.packages(c("markovchain", "igraph", "expm"))

---
## Parte 1: Miniexploración a mano — el cliente de un banco digital

Estados mensuales: **A**ctivo, **O**casional, **I**nactivo y **F**uga. Matriz de transición mensual:

$$P=\begin{array}{c|cccc} & A & O & I & F\\\hline A & 0.80 & 0.12 & 0.05 & 0.03\\ O & 0.30 & 0.45 & 0.15 & 0.10\\ I & 0.10 & 0.15 & 0.50 & 0.25\\ F & 0.05 & 0.00 & 0.05 & 0.90\end{array}$$

### 1.1 Cada fila suma 1 y diagrama de la cadena

| Fila | Suma |
|---|---|
| A | $0.80+0.12+0.05+0.03=1$ |
| O | $0.30+0.45+0.15+0.10=1$ |
| I | $0.10+0.15+0.50+0.25=1$ |
| F | $0.05+0.00+0.05+0.90=1$ |

Aristas del diagrama (cada flecha es una transición con probabilidad positiva; los lazos son las probabilidades de quedarse):

| Desde | Hacia | Probabilidad |
|---|---|---|
| A | A (lazo), O, I, F | 0.80, 0.12, 0.05, 0.03 |
| O | A, O (lazo), I, F | 0.30, 0.45, 0.15, 0.10 |
| I | A, O, I (lazo), F | 0.10, 0.15, 0.50, 0.25 |
| F | A, I, F (lazo) | 0.05, 0.05, 0.90 (no hay flecha F → O porque $P_{FO}=0$) |

Para dibujarlo con cuidado (y con las probabilidades en las flechas) está el código de la parte 2.

### 1.2 Lectura de $P_{OA}$ y $P_{FA}$ en lenguaje de negocio

- $P_{OA}=0.30$: de los clientes **ocasionales** este mes, el **30 %** estará **activo** el mes siguiente (reactivación espontánea).
- $P_{FA}=0.05$: de los clientes en **fuga** este mes, el **5 %** vuelve a ser **activo** el mes siguiente (clientes recuperados).

### 1.3 ¿Irreducible? ¿Aperiódica?

- **Irreducible: sí.** Desde cualquier estado se llega a cualquier otro. Desde A hay flechas directas a O, I y F, y desde F se vuelve a A con $P_{FA}=0.05>0$; por lo tanto cada estado alcanza a A y A alcanza a todos.
- **Aperiódica: sí.** La diagonal es positiva (hay lazos en todos los estados, p. ej. $P_{AA}=0.80$), así que existen caminos de longitud 1 de un estado a sí mismo y el período de cada estado es 1.

### 1.4 Cliente activo hoy: $\pi_0=(1,0,0,0)$

$$\pi_1=\pi_0P=(0.80,\;0.12,\;0.05,\;0.03)$$

$$\pi_2(F)=\sum_j \pi_1(j)\,P_{jF}=0.80(0.03)+0.12(0.10)+0.05(0.25)+0.03(0.90)=0.024+0.012+0.0125+0.027=\mathbf{0.0755}$$

Un cliente activo hoy tiene cerca de **7.6 %** de probabilidad de estar en fuga dentro de dos meses.

### 1.5 Predicción (sin calcular)

**Predicción del grupo:** el estado con mayor proporción a largo plazo será **Fuga**. Razones: es el estado más "pegajoso" ($P_{FF}=0.90$, permanencia esperada de $1/(1-0.90)=10$ meses, contra 5 meses en Activo) y todos los demás estados le envían probabilidad, mientras que casi no regresa nada desde Fuga. Se compara con el resultado en la parte 2.

---
## Parte 2: La cadena en R

In [ ]:
library(markovchain); library(expm)

estados <- c("Activo", "Ocasional", "Inactivo", "Fuga")
P <- matrix(c(0.80, 0.12, 0.05, 0.03,
              0.30, 0.45, 0.15, 0.10,
              0.10, 0.15, 0.50, 0.25,
              0.05, 0.00, 0.05, 0.90),
            nrow = 4, byrow = TRUE, dimnames = list(estados, estados))

rowSums(P)                                        # cada fila suma 1
mc <- new("markovchain", states = estados, transitionMatrix = P, name = "Cliente")

In [ ]:
# Comprobar a mano lo argumentado en la parte 1
is.irreducible(mc)           # TRUE
period(mc)                   # 1
summary(mc)                  # una sola clase recurrente, sin estados transitorios ni absorbentes

pi0 <- c(1, 0, 0, 0)
pi0 %*% P                    # pi_1 = 0.80 0.12 0.05 0.03
round(P %^% 2, 4)            # fila Activo: 0.6825 0.1575 0.0845 0.0755
round((pi0 %*% (P %^% 2))[4], 4)   # P(Fuga en 2 meses) = 0.0755

In [ ]:
# Distribución estacionaria
round(steadyStates(mc), 4)   # Activo 0.3218  Ocasional 0.1000  Inactivo 0.1091  Fuga 0.4692

# Comprobación: las filas de P^n convergen a pi (aquí con n = 100)
round(P %^% 100, 4)

In [ ]:
# Un cliente durante 1000 meses
set.seed(2026)
meses <- markovchainSequence(n = 1000, markovchain = mc, t0 = "Activo")
round(table(factor(meses, levels = estados)) / 1000, 3)   # cercana, pero no igual, a pi

In [ ]:
# Diagrama de la cadena con las probabilidades sobre las flechas (la 1.ª tarea de la parte 1)
library(igraph)
trans <- which(P > 0, arr.ind = TRUE)
gm <- graph_from_data_frame(data.frame(from = estados[trans[, 1]], to = estados[trans[, 2]],
                                       p = P[trans]), vertices = estados, directed = TRUE)
plot(gm, edge.label = E(gm)$p, edge.curved = 0.25, edge.arrow.size = 0.5,
     vertex.size = 35, vertex.color = "#FF871E", vertex.label.color = "#262626",
     edge.label.cex = 0.8, layout = layout_in_circle)

**Pregunta para discutir.**

- **¿Coincide con la predicción?** Sí: la distribución estacionaria es $\pi\approx(0.322,\ 0.100,\ 0.109,\ 0.469)$, con **Fuga** como el estado más grande. Casi la mitad de los clientes termina en Fuga, aunque hoy el cliente de la fila A solo tiene 3 % de irse en un mes.
- **¿Por qué pesa tanto $P_{FF}=0.90$?** Porque la fuga es casi un estado absorbente: se sale de ella con probabilidad $0.10$ por mes, mientras que entra desde los otros tres estados (3 %, 10 % y 25 % mensual). En equilibrio el flujo que entra a F debe igualar al que sale: $0.10\,\pi_F=0.03\pi_A+0.10\pi_O+0.25\pi_I$, y con una salida tan pequeña $\pi_F$ debe ser grande para compensar. En términos de tiempo: un cliente en Fuga se queda 10 meses en promedio; uno Activo, 5.
- **¿Por qué la simulación da algo parecido pero no igual?** Porque 1000 meses de **un solo cliente** son una muestra finita y **autocorrelacionada** (cada mes depende del anterior), con lo que el tamaño efectivo de muestra es mucho menor que 1000; además la trayectoria **arranca en Activo**, no en $\pi$, y ese transitorio inicial sesga un poco las frecuencias. Por el teorema ergódico, las frecuencias convergen a $\pi$ cuando el número de meses tiende a infinito, y cambian de una semilla a otra.

---
## Parte 3: Una campaña de retención

La campaña cambiaría la fila **Ocasional** de $(0.30,\,0.45,\,0.15,\,0.10)$ a $(0.40,\,0.45,\,0.10,\,0.05)$.

**1. Predicción antes de correr el código.** La campaña sube 10 puntos la reactivación (O→A) y recorta a la mitad la fuga directa de O (10 % → 5 %) y baja 5 puntos O→I. Como Ocasional es solo ~10 % de la cartera, esperamos un efecto moderado: la fuga a largo plazo debería bajar **unos 4–6 puntos porcentuales** (de ~47 % a ~42 %).

In [ ]:
P2 <- P
P2["Ocasional", ] <- c(0.40, 0.45, 0.10, 0.05)
mc2 <- new("markovchain", states = estados, transitionMatrix = P2)

comparacion <- rbind(actual  = steadyStates(mc)[1, ],
                     campana = steadyStates(mc2)[1, ])
round(comparacion, 4)
# actual   0.3218 0.1000 0.1091 0.4692
# campana  0.3724 0.1087 0.1008 0.4181

# Clientes menos en Fuga en una cartera de 200 000
clientes <- 200000
(comparacion["actual", "Fuga"] - comparacion["campana", "Fuga"]) * clientes   # ~ 10 224

**2. Resultado.** La fuga de largo plazo baja de **46.9 % a 41.8 %**: unos **5.1 puntos**, de modo que la predicción era razonable. La proporción de Activos sube de 32.2 % a 37.2 %.

**3. En una cartera de 200 000 clientes:** $(0.4692-0.4181)\times 200\,000\approx$ **10 200 clientes menos en Fuga** a largo plazo.

### Para discutir: ¿actuar sobre Inactivo o sobre la probabilidad de volver desde Fuga?

Se prueban dos alternativas y se comparan con la campaña original.

In [ ]:
# Función auxiliar: distribución estacionaria de una matriz (normalizando filas por seguridad)
estacionaria <- function(M) {
  M <- M / rowSums(M)
  steadyStates(new("markovchain", states = estados, transitionMatrix = M))[1, ]
}

# Alternativa 1: campaña sobre Inactivo, fila (0.10, 0.15, 0.50, 0.25) -> (0.15, 0.20, 0.50, 0.15)
P3 <- P;  P3["Inactivo", ] <- c(0.15, 0.20, 0.50, 0.15)

# Alternativa 2: recuperar clientes en Fuga, fila (0.05, 0.00, 0.05, 0.90) -> (0.10, 0.00, 0.05, 0.85)
P4 <- P;  P4["Fuga", ] <- c(0.10, 0.00, 0.05, 0.85)

tabla <- rbind(actual            = estacionaria(P),
               camp_Ocasional    = estacionaria(P2),
               camp_Inactivo     = estacionaria(P3),
               recuperar_Fuga    = estacionaria(P4))
tabla <- cbind(round(tabla, 4), menos_fuga_200k = round((tabla["actual", "Fuga"] - tabla[, "Fuga"]) * 200000))
tabla
# actual           0.3218 0.1000 0.1091 0.4692       0
# camp_Ocasional   0.3724 0.1087 0.1008 0.4181   10224
# camp_Inactivo    0.3661 0.1209 0.1129 0.4001   13820
# recuperar_Fuga   0.4141 0.1211 0.1129 0.3518   23476

**Lectura.** Con cambios de magnitud comparable (5–10 puntos en una fila):

| Intervención | Fuga a largo plazo | Clientes menos en Fuga (200 000) |
|---|---|---|
| Campaña Ocasional (original) | 41.8 % | ≈ 10 200 |
| Campaña Inactivo (fuga 25 % → 15 %, activación 10 % → 15 %) | 40.0 % | ≈ 13 800 |
| Recuperar Fuga (volver a Activo 5 % → 10 %) | 35.2 % | ≈ 23 500 |

Actuar sobre la fila de **Fuga** rinde más porque ese estado contiene casi la mitad de la cartera: un cambio pequeño en su fila afecta a mucha más masa de probabilidad que el mismo cambio en un estado que tiene el 10 % de los clientes. Hay que ponerlo en contexto de negocio: recuperar a un cliente que ya se fue (*win-back*) suele ser más caro y menos probable que retener a uno que todavía usa la cuenta, por lo que lo que importa es el **efecto por colón invertido**, no solo el efecto sobre $\pi$. Los números dependen de las probabilidades supuestas para cada alternativa.

---
## Parte 4: PageRank de un sitio de trámites

Siete páginas enlazadas entre sí; un navegante hace clic al azar. El PageRank es la distribución estacionaria de la cadena "seguir un enlace al azar con probabilidad $d$ / saltar a una página cualquiera con probabilidad $1-d$":

$$\pi=d\,\pi P+\frac{1-d}{n}\mathbf{1}$$

In [ ]:
library(igraph)

# aristas en pares: origen, destino
aristas <- c("Inicio","Tramites",   "Inicio","Noticias",  "Inicio","Pagos",
             "Tramites","Pagos",    "Tramites","Preguntas",
             "Noticias","Inicio",   "Noticias","Blog",    "Blog","Noticias",
             "Pagos","Inicio",      "Preguntas","Tramites",
             "Preguntas","Contacto","Contacto","Inicio")
g <- graph_from_edgelist(matrix(aristas, ncol = 2, byrow = TRUE), directed = TRUE)

pr85 <- page_rank(g, damping = 0.85)$vector
pr50 <- page_rank(g, damping = 0.50)$vector
orden <- names(sort(pr85, decreasing = TRUE))
round(cbind(d085 = pr85[orden], d050 = pr50[orden]), 4)
#           d085   d050
# Inicio   0.2831 0.2345
# Noticias 0.1876 0.1671
# Pagos    0.1591 0.1447
# Tramites 0.1352 0.1369
# Blog     0.1012 0.1132
# Preguntas 0.0789 0.1057
# Contacto 0.0549 0.0978

plot(g, vertex.size = 15 + 120 * pr85, vertex.color = "#FF871E",
     vertex.label.color = "#262626", edge.arrow.size = 0.4)

**2. Con $d=0.85$**, Inicio concentra cerca del **28 %** del PageRank, seguida de Noticias (19 %) y Pagos (16 %); Contacto apenas recibe **5.5 %**. Inicio es la página "central": recibe enlaces de Noticias, Pagos y Contacto y reparte hacia tres páginas.

**3. Con $d=0.5$** las páginas se parecen mucho más: Inicio baja a ~23 % y Contacto sube a casi 10 %. Con $d$ menor el navegante **salta a una página al azar** la mitad de las veces (en lugar del 15 %), así que la estructura de enlaces pesa menos y la distribución se acerca a la uniforme ($1/7\approx14\,\%$). En el extremo $d=0$ todas las páginas tendrían $1/7$; con $d\to1$ el resultado depende solo del grafo.

**¿Qué valor de $d$ describe mejor a un usuario real?** El número esperado de clics antes de "saltar" es $1/(1-d)$: 6.7 clics con $d=0.85$ y solo 2 con $d=0.5$. Para un sitio de trámites, donde la gente suele llegar con un objetivo concreto (buscador, marcador o enlace directo a Pagos o Trámites) y se va tras dos o tres clics, una $d$ intermedia-baja (0.5–0.7) es más realista que el 0.85 clásico de Google, que modela navegación larga y exploratoria. Lo ideal es **calibrar $d$ con datos**: $d\approx 1-1/(\text{clics promedio por sesión})$.

In [ ]:
# 4. Un enlace nuevo que suba el PageRank de Contacto: Pagos -> Contacto
# (Pagos tiene mucho PageRank y hoy solo reparte hacia Inicio)
g2 <- add_edges(g, match(c("Pagos", "Contacto"), V(g)$name))
pr85_2 <- page_rank(g2, damping = 0.85)$vector

round(cbind(antes = pr85[orden], despues = pr85_2[orden]), 4)
# Contacto sube de 0.0549 a 0.1166 (más del doble); Inicio baja de 0.2831 a 0.2590.

# Comparación con otras opciones
enlace <- function(origen) {
  gx <- add_edges(g, match(c(origen, "Contacto"), V(g)$name))
  unname(round(page_rank(gx, damping = 0.85)$vector["Contacto"], 4))
}
sapply(c("Inicio", "Noticias", "Pagos", "Blog"), enlace)
# Inicio 0.1165  Noticias 0.1010  Pagos 0.1166  Blog 0.0905

**4. Propuesta:** agregar el enlace **Pagos → Contacto** (y/o **Inicio → Contacto**). El PageRank de Contacto pasa de **5.5 % a ≈ 11.7 %**, más del doble. Funciona porque el PageRank que una página entrega se **reparte entre sus enlaces salientes** y las páginas de origen (Pagos e Inicio) son de las que más PageRank tienen; enlazar desde una página con menos PageRank (p. ej. Blog, que lleva Contacto a 9.1 %) sirve menos. El costo es que las demás páginas ceden una parte (la suma sigue siendo 1): el enlace nuevo le quita PageRank a la página que antes recibía todo el flujo saliente de Pagos (Inicio).

---
## Parte 5: Reflexión y tema del proyecto

**Observación matemática.** Lo que más sorprende de la distribución estacionaria del banco es que **casi la mitad de los clientes termina en Fuga** (46.9 %) aunque un cliente Activo solo tiene 3 % de irse en un mes: la dinámica de largo plazo depende del conjunto de la matriz, y una diagonal de 0.90 en un estado al que llegan los demás lo vuelve casi absorbente. También sorprende que cambios pequeños en una fila (campaña de 5–10 puntos) muevan la cartera entre 10 000 y 23 000 clientes.

**Observación técnica.** En R, `markovchain` trae objetos y funciones propias para el análisis (`steadyStates`, `is.irreducible`, `period`, `markovchainSequence`) y trabaja con matrices con nombres de estados, lo que hace el código muy cercano a la notación matemática; en Python normalmente se hace con `numpy` (autovectores de $P^T$, `np.linalg.matrix_power`) y la simulación se programa a mano. La contraparte en R es la instalación de paquetes en Colab, que tarda y hay que repetir al reiniciar el entorno, y la indexación desde 1 (en Python desde 0).

**Pregunta abierta.** ¿Puede nuestro proyecto plantearse, total o parcialmente, como una cadena de Markov? _(completar según el carril del grupo: ¿cuáles serían los estados, con qué frecuencia se observa la transición y de dónde saldría la matriz $P$ —conteo de transiciones en datos históricos—? ¿Es razonable la propiedad de Markov, es decir, que el siguiente estado dependa solo del actual?)_

**Carril elegido para el proyecto integrador:** _(escribir aquí el carril confirmado por correo al profesor; los temas no se repiten y se asignan por orden de llegada)_